<a href="https://colab.research.google.com/github/Musfik14/SMR-project1/blob/main/1st_try_with_full_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# # ======================================================
# # FR_00 (RUN ONCE per new Colab runtime)
# # - Mount Drive
# # - Create run folders under asvspoof_project
# # - Download ASVspoof2019 from KaggleHub to Colab SSD
# # - Set correct paths + GPU check
# # ======================================================

# !pip -q install kagglehub==0.2.9

# from google.colab import drive
# import os, time, json, shutil
# import torch
# import kagglehub

# # ---- 1) Mount Drive ----
# drive.mount("/content/drive")

# # ---- 2) Project root (YOUR existing folder) ----
# PROJECT_DIR = "/content/drive/MyDrive/asvspoof_project"

# # ---- 3) Create a fresh RUN folder (inside your project) ----
# RUN_ID  = time.strftime("run_%Y%m%d_%H%M%S")
# RUN_DIR = os.path.join(PROJECT_DIR, "runs", RUN_ID)

# DIRS = {
#     "run": RUN_DIR,
#     "ckpt": os.path.join(RUN_DIR, "checkpoints"),
#     "feat": os.path.join(RUN_DIR, "features"),
#     "scores": os.path.join(RUN_DIR, "scores"),
#     "logs": os.path.join(RUN_DIR, "logs"),
#     "plots": os.path.join(RUN_DIR, "plots"),
#     "meta": os.path.join(RUN_DIR, "meta"),
# }
# for p in DIRS.values():
#     os.makedirs(p, exist_ok=True)

# # ---- 4) Download dataset to Colab local SSD (fast training) ----
# # KaggleHub dataset: awsaf49/asvpoof-2019-dataset  (your earlier usage)
# src_path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
# DATA_DIR = "/content/ASVspoof2019_RAW"  # local SSD target

# # Copy only if not already present (prevents re-copy on rerun)
# if not os.path.exists(DATA_DIR):
#     shutil.copytree(src_path, DATA_DIR)

# # ---- 5) GPU check ----
# print("CUDA available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))

# # ---- 6) Save minimal metadata (Drive) ----
# CFG = {
#     "run_id": RUN_ID,
#     "project_dir": PROJECT_DIR,
#     "data_dir": DATA_DIR,                 # SSD path
#     "dirs": DIRS,
#     "dataset_source": "kagglehub: awsaf49/asvpoof-2019-dataset",
#     "protocols": ["LA", "PA"],
#     "resume_enabled": True,
#     "seed": 42
# }
# with open(os.path.join(DIRS["meta"], "config.json"), "w") as f:
#     json.dump(CFG, f, indent=2)

# # ---- 7) Sanity checks ----
# print("RUN_ID:", RUN_ID)
# print("RUN_DIR:", RUN_DIR)
# print("DATA_DIR exists:", os.path.exists(DATA_DIR))
# print("LA exists:", os.path.exists(os.path.join(DATA_DIR, "LA")))
# print("PA exists:", os.path.exists(os.path.join(DATA_DIR, "PA")))
# print("Config:", os.path.join(DIRS["meta"], "config.json"))


Mounted at /content/drive


100%|██████████| 23.6G/23.6G [03:57<00:00, 107MB/s]

Extracting model files...


CUDA available: True
GPU: Tesla T4
RUN_ID: run_20251227_111410
RUN_DIR: /content/drive/MyDrive/asvspoof_project/runs/run_20251227_111410
DATA_DIR exists: True
LA exists: True
PA exists: True
Config: /content/drive/MyDrive/asvspoof_project/runs/run_20251227_111410/meta/config.json


In [20]:
# ======================================================
# FR/RS_00_RESUME_SAFE
# - Fresh OR Resume (explicit)
# ======================================================

from google.colab import drive
import os, json, shutil, torch
import kagglehub

# ---- 1) Mount Drive ----
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/asvspoof_project"
RUNS_DIR = os.path.join(PROJECT_DIR, "runs")

# =========================
# 🔴 USER MUST SET THIS
# =========================
MODE = "resume"   # "fresh" or "resume"
RUN_ID = "run_20251227_111410"   # required for resume
# =========================

assert MODE in ["fresh", "resume"]

# ---- 2) Handle RUN_ID ----
if MODE == "fresh":
    import time
    RUN_ID = time.strftime("run_%Y%m%d_%H%M%S")
    print("🆕 Fresh run created:", RUN_ID)
else:
    assert RUN_ID is not None, "RUN_ID must be provided for resume"
    print("🔁 Resuming run:", RUN_ID)

RUN_DIR = os.path.join(RUNS_DIR, RUN_ID)

DIRS = {
    "run": RUN_DIR,
    "ckpt": os.path.join(RUN_DIR, "checkpoints"),
    "feat": os.path.join(RUN_DIR, "features"),
    "scores": os.path.join(RUN_DIR, "scores"),
    "logs": os.path.join(RUN_DIR, "logs"),
    "plots": os.path.join(RUN_DIR, "plots"),
    "meta": os.path.join(RUN_DIR, "meta"),
}

for p in DIRS.values():
    os.makedirs(p, exist_ok=True)   # SAFE: no overwrite

# ---- 3) Dataset on SSD (safe reuse) ----
DATA_DIR = "/content/ASVspoof2019_RAW"
if not os.path.exists(DATA_DIR):
    src_path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
    shutil.copytree(src_path, DATA_DIR)

# ---- 4) GPU check ----
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ---- 5) Load or save config ----
cfg_path = os.path.join(DIRS["meta"], "config.json")

if MODE == "fresh":
    CFG = {
        "run_id": RUN_ID,
        "project_dir": PROJECT_DIR,
        "data_dir": DATA_DIR,
        "dirs": DIRS,
        "protocols": ["LA", "PA"],
        "resume_enabled": True,
        "seed": 42
    }
    with open(cfg_path, "w") as f:
        json.dump(CFG, f, indent=2)
else:
    with open(cfg_path, "r") as f:
        CFG = json.load(f)

print("✅ RUN_ID:", RUN_ID)
print("✅ Using run directory:", RUN_DIR)
print("✅ DATA_DIR:", DATA_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔁 Resuming run: run_20251227_111410
CUDA available: True
GPU: Tesla T4
✅ RUN_ID: run_20251227_111410
✅ Using run directory: /content/drive/MyDrive/asvspoof_project/runs/run_20251227_111410
✅ DATA_DIR: /content/ASVspoof2019_RAW


In [2]:
# ======================================================
# FR_01_FR/RS: Locked dependencies for reproducibility
# ======================================================

!pip install -q \
    torch==2.1.2 \
    librosa==0.10.1 \
    soundfile==0.12.1 \
    numpy==1.24.4 \
    scipy==1.11.4 \
    matplotlib==3.8.2 \
    scikit-learn==1.3.2 \
    pandas==2.1.4 \
    tqdm==4.66.1

print("FR_01 completed: dependencies installed and locked.")


ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==2.1.2
FR_01 completed: dependencies installed and locked.


In [4]:
# ======================================================
# FR/RS_02_PROTOCOLS: Parse LA/PA protocols + balance index
# ======================================================
import os
import pandas as pd

DATA_DIR = "/content/ASVspoof2019_RAW"

def find_file(root, name):
    for r, _, f in os.walk(root):
        if name in f:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

def load_proto(proto, split):
    base = os.path.join(DATA_DIR, proto)
    fname = {
        "train": f"ASVspoof2019.{proto}.cm.train.trn.txt",
        "dev":   f"ASVspoof2019.{proto}.cm.dev.trl.txt",
    }[split]
    path = find_file(base, fname)
    df = pd.read_csv(path, sep=" ", header=None)
    df = df[[1,4]]                  # file_id, label
    df.columns = ["file_id","label"]
    df["label"] = df["label"].map({"bonafide":0,"spoof":1})
    df["proto"] = proto
    return df

# ---- Load LA + PA ----
train_df = pd.concat([load_proto("LA","train"), load_proto("PA","train")])
dev_df   = pd.concat([load_proto("LA","dev"),   load_proto("PA","dev")])

train_df = train_df.reset_index(drop=True)
dev_df   = dev_df.reset_index(drop=True)

# ---- Show imbalance (important for next steps) ----
print("Train distribution:\n", train_df.label.value_counts(normalize=True))
print("Dev distribution:\n", dev_df.label.value_counts(normalize=True))

print("FR_02 done: protocol index ready (no features yet).")


Train distribution:
 label
1    0.899471
0    0.100529
Name: proportion, dtype: float64
Dev distribution:
 label
1    0.854283
0    0.145717
Name: proportion, dtype: float64
FR_02 done: protocol index ready (no features yet).


In [5]:
# ======================================================
# FR/RS_03: Compute class weights & sampler
# ======================================================
import torch
from torch.utils.data import WeightedRandomSampler
import numpy as np

# ---- Compute class counts ----
labels = train_df["label"].values
class_counts = np.bincount(labels)     # [bonafide, spoof]

# ---- Inverse-frequency weights ----
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels]

print("Class counts:", class_counts)
print("Class weights:", class_weights)

# ---- Weighted sampler (use in DataLoader) ----
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("FR_03 done: balanced sampler ready (no augmentation yet).")


Class counts: [ 7980 71400]
Class weights: [1.25313283e-04 1.40056022e-05]
FR_03 done: balanced sampler ready (no augmentation yet).


In [6]:
# ======================================================
# FR_04_FEATURE_PLAN: Paths + extraction params (NO compute)
# ======================================================
import os, json

# ---- Cache locations ----
MEL_CACHE_DIR = os.path.join(DIRS["feat"], "logmel")     # Drive
CUE_CACHE_DIR = f"/content/cues_cache/{RUN_ID}"          # SSD

os.makedirs(MEL_CACHE_DIR, exist_ok=True)
os.makedirs(CUE_CACHE_DIR, exist_ok=True)

# ---- Feature parameters (locked for reproducibility) ----
FEATURE_CFG = {
    "sr": 16000,
    "n_mels": 80,
    "n_fft": 1024,
    "hop": 160,
    "win": 400,
    "logmel_power": 2.0,
    "cue_dim": 9,          # spectral flux (4) + phase (3) + energy (2)
    "storage": {
        "logmel": "drive",
        "cues": "ssd"
    }
}

# ---- Save feature config (resume-safe) ----
with open(os.path.join(DIRS["meta"], "feature_config.json"), "w") as f:
    json.dump(FEATURE_CFG, f, indent=2)

print("FR_04 done: feature & cue plan fixed and saved.")
print("Log-Mel cache:", MEL_CACHE_DIR)
print("Cue cache:", CUE_CACHE_DIR)


FR_04 done: feature & cue plan fixed and saved.
Log-Mel cache: /content/drive/MyDrive/asvspoof_project/runs/run_20251227_111410/features/logmel
Cue cache: /content/cues_cache/run_20251227_111410


In [19]:
# ======================================================
# FR_05_LOGMEL_PROTO (FIXED AUDIO PATH)
# ======================================================
import os, json, warnings
import numpy as np
import librosa
from tqdm import tqdm

warnings.filterwarnings("ignore")

with open(os.path.join(DIRS["meta"], "feature_config.json"), "r") as f:
    FCFG = json.load(f)

SR   = FCFG["sr"]
N_M  = FCFG["n_mels"]
N_F  = FCFG["n_fft"]
HOP  = FCFG["hop"]
WIN  = FCFG["win"]

def audio_path(file_id, proto, split):
    return os.path.join(
        DATA_DIR, proto, proto,
        f"ASVspoof2019_{proto}_{split}",
        "flac",
        f"{file_id}.flac"
    )

def extract_logmel(path):
    y, _ = librosa.load(path, sr=SR)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_F,
        hop_length=HOP, win_length=WIN,
        n_mels=N_M
    )
    return librosa.power_to_db(mel).astype(np.float32)

def run_logmel(df, split):
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"LogMel {split}"):
        out_dir = os.path.join(MEL_CACHE_DIR, r.proto, split)
        os.makedirs(out_dir, exist_ok=True)

        out = os.path.join(out_dir, r.file_id + ".npy")
        if os.path.exists(out):
            continue

        try:
            feat = extract_logmel(audio_path(r.file_id, r.proto, split))
            np.save(out, feat)
        except Exception:
            pass

run_logmel(train_df, "train")
run_logmel(dev_df, "dev")

print("✅ FR_05 DONE: Log-Mel extracted correctly.")


LogMel dev: 100%|██████████| 54544/54544 [56:50<00:00, 15.99it/s]

✅ FR_05 DONE: Log-Mel extracted correctly.


In [7]:
# ======================================================
# FR/RS_06_CUES_PROTO (FIXED AUDIO PATH)
# ======================================================
import os, json, warnings
import numpy as np
import librosa
from tqdm import tqdm
from scipy.stats import skew

warnings.filterwarnings("ignore")

with open(os.path.join(DIRS["meta"], "feature_config.json"), "r") as f:
    FCFG = json.load(f)

SR  = FCFG["sr"]
N_F = FCFG["n_fft"]
HOP = FCFG["hop"]

def audio_path(file_id, proto, split):
    return os.path.join(
        DATA_DIR, proto, proto,
        f"ASVspoof2019_{proto}_{split}",
        "flac",
        f"{file_id}.flac"
    )

def extract_cues(path):
    y, _ = librosa.load(path, sr=SR)
    S = np.abs(librosa.stft(y, n_fft=N_F, hop_length=HOP))
    flux = np.sqrt(np.sum(np.diff(S, axis=1)**2, axis=0))
    rms = librosa.feature.rms(y=y, hop_length=HOP)[0]
    phase = np.unwrap(np.angle(librosa.stft(y)), axis=1)
    pd = np.mean(np.abs(np.diff(phase, axis=1)), axis=0)

    return np.array([
        flux.mean(), flux.std(), flux.max(), np.percentile(flux,95),
        pd.mean(), pd.std(), skew(pd),
        rms.mean(), rms.std()
    ], dtype=np.float32)

def run_cues(df, split):
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Cues {split}"):
        out_dir = os.path.join(CUE_CACHE_DIR, r.proto, split)
        os.makedirs(out_dir, exist_ok=True)

        out = os.path.join(out_dir, r.file_id + ".npz")
        if os.path.exists(out):
            continue

        try:
            cue = extract_cues(audio_path(r.file_id, r.proto, split))
            np.savez(out, cue=cue)
        except Exception:
            pass

run_cues(train_df, "train")
run_cues(dev_df, "dev")

print("✅ FR_06 DONE: cues extracted correctly.")


Cues dev: 100%|██████████| 54544/54544 [40:00<00:00, 22.72it/s]

✅ FR_06 DONE: cues extracted correctly.


In [8]:
# ======================================================
# FR/RS_07 : Hybrid dataset + collate (final)
# ======================================================
import os, json, numpy as np, torch
from torch.utils.data import Dataset
import torch.nn.functional as F

# ---- Reload feature config (resume-safe) ----
with open(os.path.join(DIRS["meta"], "feature_config.json"), "r") as f:
    FCFG = json.load(f)

class HybridDataset(Dataset):
    def __init__(self, df, split):
        self.df = df.reset_index(drop=True)
        self.split = split

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]

        mel_path = os.path.join(
            MEL_CACHE_DIR, r.proto, self.split, r.file_id + ".npy"
        )
        cue_path = os.path.join(
            CUE_CACHE_DIR, r.proto, self.split, r.file_id + ".npz"
        )

        if not os.path.exists(mel_path):
            raise FileNotFoundError(mel_path)
        if not os.path.exists(cue_path):
            raise FileNotFoundError(cue_path)

        mel = torch.tensor(np.load(mel_path), dtype=torch.float32).unsqueeze(0)
        cue = torch.tensor(np.load(cue_path)["cue"], dtype=torch.float32)
        label = torch.tensor(r.label, dtype=torch.float32)

        return mel, cue, label

def collate_pad(batch):
    mels, cues, labels = zip(*batch)
    T = max(m.shape[-1] for m in mels)
    mels = [F.pad(m, (0, T - m.shape[-1])) for m in mels]
    return (
        torch.stack(mels),
        torch.stack(cues),
        torch.stack(labels).unsqueeze(1)
    )

print("✅ FR_07 done: proto-aware dataset + collate ready.")


✅ FR_07 done: proto-aware dataset + collate ready.


In [21]:
# # ======================================================
# # TEST_01_FEATURE_CHECK: Verify feature locations & coverage
# # ======================================================
# import os
# import random

# def count_files(root):
#     if not os.path.exists(root):
#         return 0
#     return len([f for f in os.listdir(root) if f.endswith(".npy")])

# print("=== LOG-MEL FEATURE STRUCTURE CHECK ===")
# for proto in ["LA", "PA"]:
#     for split in ["train", "dev"]:
#         path = os.path.join(MEL_CACHE_DIR, proto, split)
#         print(f"{proto}/{split} exists:", os.path.exists(path),
#               "| files:", count_files(path))

# print("\n=== CUE FEATURE STRUCTURE CHECK ===")
# for proto in ["LA", "PA"]:
#     for split in ["train", "dev"]:
#         path = os.path.join(CUE_CACHE_DIR, proto, split)
#         print(f"{proto}/{split} exists:", os.path.exists(path),
#               "| files:", len(os.listdir(path)) if os.path.exists(path) else 0)

# print("\n=== RANDOM SAMPLE LOOKUP TEST ===")
# sample_rows = train_df.sample(10, random_state=42)

# for _, r in sample_rows.iterrows():
#     mel_path = os.path.join(MEL_CACHE_DIR, r.proto, "train", r.file_id + ".npy")
#     cue_path = os.path.join(CUE_CACHE_DIR, r.proto, "train", r.file_id + ".npz")
#     print(r.proto, r.file_id,
#           "mel:", os.path.exists(mel_path),
#           "cue:", os.path.exists(cue_path))


=== LOG-MEL FEATURE STRUCTURE CHECK ===
LA/train exists: True | files: 25380
LA/dev exists: True | files: 24844
PA/train exists: True | files: 54000
PA/dev exists: True | files: 29700

=== CUE FEATURE STRUCTURE CHECK ===
LA/train exists: True | files: 25380
LA/dev exists: True | files: 24844
PA/train exists: True | files: 54000
PA/dev exists: True | files: 29700

=== RANDOM SAMPLE LOOKUP TEST ===
LA LA_T_7107224 mel: True cue: True
PA PA_T_0027345 mel: True cue: True
LA LA_T_8980556 mel: True cue: True
LA LA_T_3031398 mel: True cue: True
PA PA_T_0009880 mel: True cue: True
PA PA_T_0032859 mel: True cue: True
PA PA_T_0026428 mel: True cue: True
LA LA_T_8184237 mel: True cue: True
LA LA_T_7623882 mel: True cue: True
PA PA_T_0050855 mel: True cue: True


In [22]:
# # ======================================================
# # TEST_02_AUDIO_PATH_DISCOVERY
# # Find where FLAC files actually live
# # ======================================================
# import os

# def find_flac_root(base, proto):
#     flac_roots = []
#     for root, dirs, files in os.walk(os.path.join(base, proto)):
#         if "flac" in root.lower():
#             flac_roots.append(root)
#     return flac_roots

# for proto in ["LA", "PA"]:
#     roots = find_flac_root(DATA_DIR, proto)
#     print(f"\n{proto} FLAC roots found ({len(roots)}):")
#     for r in roots[:5]:
#         print(" ", r)



LA FLAC roots found (3):
  /content/ASVspoof2019_RAW/LA/LA/ASVspoof2019_LA_train/flac
  /content/ASVspoof2019_RAW/LA/LA/ASVspoof2019_LA_dev/flac
  /content/ASVspoof2019_RAW/LA/LA/ASVspoof2019_LA_eval/flac

PA FLAC roots found (3):
  /content/ASVspoof2019_RAW/PA/PA/ASVspoof2019_PA_eval/flac
  /content/ASVspoof2019_RAW/PA/PA/ASVspoof2019_PA_dev/flac
  /content/ASVspoof2019_RAW/PA/PA/ASVspoof2019_PA_train/flac


In [23]:
# # ======================================================
# # TEST_03_AUDIO_PATH_FIXED
# # ======================================================
# import os

# def audio_path_fixed(file_id, proto, split):
#     return os.path.join(
#         DATA_DIR, proto, proto,
#         f"ASVspoof2019_{proto}_{split}",
#         "flac",
#         f"{file_id}.flac"
#     )

# # test on few random samples
# samples = train_df.sample(5, random_state=0)
# for _, r in samples.iterrows():
#     p = audio_path_fixed(r.file_id, r.proto, "train")
#     print(r.proto, r.file_id, "→ exists:", os.path.exists(p))


LA LA_T_1323302 → exists: True
LA LA_T_9980379 → exists: True
LA LA_T_5045892 → exists: True
PA PA_T_0006524 → exists: True
LA LA_T_1377238 → exists: True


In [9]:
# ======================================================
# FR/RS_08A_MODEL_DEF: Lightweight CNN + Cue Fusion
# ======================================================
import torch
import torch.nn as nn

class LightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(32, 16)

    def forward(self, x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class CueMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )

    def forward(self, x):
        return self.net(x)

class FusionNet(nn.Module):
    def __init__(self, cue_dim):
        super().__init__()
        self.cnn = LightCNN()
        self.cue = CueMLP(cue_dim)
        self.classifier = nn.Linear(16 + 8, 1)

    def forward(self, mel, cue):
        f_mel = self.cnn(mel)
        f_cue = self.cue(cue)
        fused = torch.cat([f_mel, f_cue], dim=1)
        return torch.sigmoid(self.classifier(fused))

print("✅ FusionNet defined and ready.")


✅ FusionNet defined and ready.


In [25]:
# ======================================================
# FR_08_TRAIN_FIRST: Fresh training with checkpoints
# ======================================================
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Dataset & Loader ----
train_ds = HybridDataset(train_df, "train")
dev_ds   = HybridDataset(dev_df, "dev")

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    sampler=train_sampler,      # class-balanced
    num_workers=0,
    collate_fn=collate_pad
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_pad
)

# ---- Model / Optimizer / Loss ----
model = FusionNet(cue_dim=FCFG["cue_dim"]).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCELoss()

EPOCHS = 10
START_EPOCH = 0
CKPT_DIR = DIRS["ckpt"]

# ---- Training Loop ----
for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for mel, cue, y in pbar:
        mel = mel.to(DEVICE)
        cue = cue.to(DEVICE)
        y   = y.to(DEVICE)

        optimizer.zero_grad()
        out = model(mel, cue)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # ---- Save FULL checkpoint (resume-safe) ----
    torch.save({
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "run_id": RUN_ID
    }, os.path.join(CKPT_DIR, f"{RUN_ID}_epoch{epoch+1}.pt"))

    print(f"✔ Saved checkpoint: epoch {epoch+1}")

print("✅ FR_08 completed: training started successfully.")


Epoch 1/10: 100%|██████████| 2481/2481 [11:41<00:00,  3.54it/s, loss=0.3545]


✔ Saved checkpoint: epoch 1


Epoch 2/10: 100%|██████████| 2481/2481 [11:08<00:00,  3.71it/s, loss=0.3238]


✔ Saved checkpoint: epoch 2


Epoch 3/10: 100%|██████████| 2481/2481 [11:05<00:00,  3.73it/s, loss=0.6485]


...

✔ Saved checkpoint: epoch 8


Epoch 9/10: 100%|██████████| 2481/2481 [11:00<00:00,  3.76it/s, loss=0.1250]


✔ Saved checkpoint: epoch 9


Epoch 10/10: 100%|██████████| 2481/2481 [10:57<00:00,  3.78it/s, loss=0.2731]

✔ Saved checkpoint: epoch 10
✅ FR_08 completed: training started successfully.


In [10]:
# # ======================================================
# # FR_09_TRAIN_SETUP: loaders + optimizer (resume-safe)
# # ======================================================
# import torch
# import torch.optim as optim
# from torch.utils.data import DataLoader

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # ---- Datasets ----
# train_ds = HybridDataset(train_df, "train")
# dev_ds   = HybridDataset(dev_df, "dev")

# # ---- DataLoaders (IMPORTANT: sampler for balance) ----
# train_loader = DataLoader(
#     train_ds,
#     batch_size=32,
#     sampler=train_sampler,     # ← class balanced
#     num_workers=0,
#     collate_fn=collate_pad
# )

# dev_loader = DataLoader(
#     dev_ds,
#     batch_size=32,
#     shuffle=False,
#     num_workers=0,
#     collate_fn=collate_pad
# )

# # ---- Model ----
# model = FusionNet(cue_dim=FCFG["cue_dim"]).to(DEVICE)

# # ---- Optimizer + Loss ----
# optimizer = optim.Adam(model.parameters(), lr=1e-3)
# criterion = torch.nn.BCELoss()

# # ---- Training metadata ----
# START_EPOCH = 0
# EPOCHS = 10

# print("FR_09 done:")
# print("• Balanced loader ready")
# print("• Model / optimizer initialized")
# print("• Resume-compatible state prepared")


FR_09 done:
• Balanced loader ready
• Model / optimizer initialized
• Resume-compatible state prepared


In [13]:
# ======================================================
# FR_10_VALIDATE: Dev-set scoring (no training)
# ======================================================
import torch, os
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Load latest checkpoint ----
ckpts = sorted([
    f for f in os.listdir(DIRS["ckpt"])
    if f.startswith(RUN_ID) and f.endswith(".pt")
])
latest_ckpt = os.path.join(DIRS["ckpt"], ckpts[-1])
print("Using checkpoint:", latest_ckpt)

ckpt = torch.load(latest_ckpt, map_location=DEVICE)

# ---- Rebuild model ----
model = FusionNet(cue_dim=FCFG["cue_dim"]).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

# ---- Dev loader ----
dev_ds = HybridDataset(dev_df, "dev")
dev_loader = DataLoader(
    dev_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_pad
)

# ---- Collect scores ----
records = []

with torch.no_grad():
    for mel, cue, y in tqdm(dev_loader, desc="Validating"):
        mel = mel.to(DEVICE)
        cue = cue.to(DEVICE)

        scores = model(mel, cue).squeeze(1).cpu().numpy()

        for s in scores:
            records.append(s)

# ---- Save scores aligned with dev_df ----
score_df = dev_df.copy()
score_df["score"] = records

out_path = os.path.join(DIRS["scores"], f"{RUN_ID}_dev_scores.csv")
score_df.to_csv(out_path, index=False)

print("✅ Validation completed.")
print("Scores saved at:", out_path)
print(score_df.head())


Using checkpoint: /content/drive/MyDrive/asvspoof_project/runs/run_20251227_111410/checkpoints/run_20251227_111410_logits_epoch3.pt


Validating: 100%|██████████| 1705/1705 [1:57:22<00:00,  4.13s/it]


✅ Validation completed.
Scores saved at: /content/drive/MyDrive/asvspoof_project/runs/run_20251227_111410/scores/run_20251227_111410_dev_scores.csv
        file_id  label proto     score
0  LA_D_1047731      0    LA  0.000034
1  LA_D_1105538      0    LA  0.000252
2  LA_D_1125976      0    LA  0.002894
3  LA_D_1293230      0    LA  0.000747
4  LA_D_1340209      0    LA  0.005572


In [14]:
# ======================================================
# FR/RS_10_METRICS: EER + min-tDCF (Dev set)
# ======================================================
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

# ---- Load dev scores ----
score_path = os.path.join(DIRS["scores"], f"{RUN_ID}_dev_scores.csv")
df = pd.read_csv(score_path)

# Convention: label 0 = bonafide, 1 = spoof
y_true = df["label"].values
y_score = df["score"].values

# ---- EER computation ----
fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2

print(f"✅ EER = {eer * 100:.2f} %")

# ---- min-tDCF (simplified ASVspoof 2019 constants) ----
# These constants are standard for LA dev evaluation
P_tar = 0.9801
P_non = 0.0099
P_spoof = 0.0100
C_miss = 1
C_fa = 10
C_fa_spoof = 10

tDCF = (
    C_miss * P_tar * fnr +
    C_fa * P_non * fpr +
    C_fa_spoof * P_spoof * fpr
)

min_tdcf = np.min(tDCF)

print(f"✅ min-tDCF = {min_tdcf:.4f}")


✅ EER = 21.42 %
✅ min-tDCF = 0.1839


In [15]:
# ======================================================
# FR_11A_MODEL_FIX: remove sigmoid (logits output)
# ======================================================
import torch
import torch.nn as nn

class FusionNet(nn.Module):
    def __init__(self, cue_dim):
        super().__init__()
        self.cnn = LightCNN()
        self.cue = CueMLP(cue_dim)
        self.classifier = nn.Linear(16 + 8, 1)

    def forward(self, mel, cue):
        f_mel = self.cnn(mel)
        f_cue = self.cue(cue)
        fused = torch.cat([f_mel, f_cue], dim=1)
        return self.classifier(fused)   # ❌ no sigmoid


In [25]:
# ======================================================
# FR_09A_DATALOADERS_ONLY: rebuild loaders (resume-safe)
# ======================================================
from torch.utils.data import DataLoader

# ---- Datasets ----
train_ds = HybridDataset(train_df, "train")
dev_ds   = HybridDataset(dev_df, "dev")

# ---- DataLoaders ----
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    sampler=train_sampler,
    num_workers=0,
    collate_fn=collate_pad
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_pad
)

print("✅ train_loader and dev_loader rebuilt.")


✅ train_loader and dev_loader rebuilt.


In [24]:
# ======================================================
# FR_05_PATCH: Re-extract ONLY missing log-Mel features
# ======================================================
import os, json, librosa, numpy as np
from tqdm import tqdm

with open(os.path.join(DIRS["meta"], "feature_config.json"), "r") as f:
    FCFG = json.load(f)

SR   = FCFG["sr"]
N_M  = FCFG["n_mels"]
N_F  = FCFG["n_fft"]
HOP  = FCFG["hop"]
WIN  = FCFG["win"]

def audio_path(file_id, proto, split):
    return os.path.join(
        DATA_DIR, proto, proto,
        f"ASVspoof2019_{proto}_{split}",
        "flac",
        f"{file_id}.flac"
    )

def extract_logmel(path):
    y, _ = librosa.load(path, sr=SR)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_F,
        hop_length=HOP, win_length=WIN,
        n_mels=N_M
    )
    return librosa.power_to_db(mel).astype(np.float32)

def patch_missing(df, split):
    missing = []
    for _, r in df.iterrows():
        out = os.path.join(MEL_CACHE_DIR, r.proto, split, r.file_id + ".npy")
        if not os.path.exists(out):
            missing.append(r)

    print(f"🔍 Missing log-mel files for {split}: {len(missing)}")

    for r in tqdm(missing, desc=f"Patching {split}"):
        out_dir = os.path.join(MEL_CACHE_DIR, r.proto, split)
        os.makedirs(out_dir, exist_ok=True)

        try:
            feat = extract_logmel(audio_path(r.file_id, r.proto, split))
            np.save(os.path.join(out_dir, r.file_id + ".npy"), feat)
        except Exception as e:
            print("❌ Failed:", r.file_id, e)

# ---- Patch both splits ----
patch_missing(train_df, "train")
patch_missing(dev_df, "dev")

print("✅ FR_05_PATCH completed.")


🔍 Missing log-mel files for train: 0


Patching train: 0it [00:00, ?it/s]


🔍 Missing log-mel files for dev: 0


Patching dev: 0it [00:00, ?it/s]

✅ FR_05_PATCH completed.


In [26]:
# ======================================================
# FR_11B_TRAIN_LOGITS: Fresh training with progress bar
# ======================================================
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Model (fresh init) ----
model = FusionNet(cue_dim=FCFG["cue_dim"]).to(DEVICE)

# ---- Loss & Optimizer ----
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
START_EPOCH = 0
CKPT_DIR = DIRS["ckpt"]

# ---- Training Loop ----
for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"FR_11B Epoch {epoch+1}/{EPOCHS}")

    for mel, cue, y in pbar:
        mel = mel.to(DEVICE)
        cue = cue.to(DEVICE)
        y   = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(mel, cue)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(
            batch_loss=f"{loss.item():.4f}",
            avg_loss=f"{running_loss / (pbar.n + 1):.4f}"
        )

    avg_epoch_loss = running_loss / len(train_loader)

    # ---- Save checkpoint ----
    torch.save({
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "run_id": RUN_ID,
        "loss_type": "BCEWithLogitsLoss",
        "avg_epoch_loss": avg_epoch_loss
    }, os.path.join(CKPT_DIR, f"{RUN_ID}_logits_epoch{epoch+1}.pt"))

    print(f"✔ Epoch {epoch+1} done | Avg Loss: {avg_epoch_loss:.4f}")

print("✅ FR_11B completed: logits-based fresh training finished.")


FR_11B Epoch 1/10:   2%|▏         | 51/2481 [08:09<6:28:42,  9.60s/it, avg_loss=0.6887, batch_loss=0.6505]


KeyboardInterrupt: 